In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker

np.random.seed(42)

DATA_PATH  = Path("../../data/merged_df_volume_subcortical_subgroups.csv")
COLOC_PATH = Path("../../results/nispace_group_comparison_results_subgroups_subcortical.csv")
OUT_DIR    = Path("../../results/figures/boxplots")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONTRAST_ORDER = ["RBD vs HC", "Hyposmia vs HC"]

MAP_INFO = {
    "Serotonin | target-5HT1a_tracer-way100635_n-35_dx-hc_pub-savli2012":       ("5-HT1a",  "5HT1a\n(WAY)",     "Serotonin"),
    "Serotonin | target-5HT1b_tracer-p943_n-23_dx-hc_pub-savli2012":            ("5-HT1b",  "5HT1b\n(P943)",    "Serotonin"),
    "Serotonin | target-5HT2a_tracer-altanserin_n-19_dx-hc_pub-savli2012":      ("5-HT2a",  "5HT2a\n(altan)",   "Serotonin"),
    "Serotonin | target-5HT4_tracer-sb207145_n-59_dx-hc_pub-beliveau2017":      ("5-HT4",   "5HT4\n(SB207)",    "Serotonin"),
    "Serotonin | target-5HTT_tracer-dasb_n-18_dx-hc_pub-savli2012":             ("5-HTT",   "5HTT\n(DASB)",     "Serotonin"),
    "Dopamine | target-D1_tracer-sch23390_n-13_dx-hc_pub-kaller2017":           ("D1",      "D1\n(SCH23)",      "Dopamine"),
    "Dopamine | target-D23_tracer-flb457_n-55_dx-hc_pub-sandiego2015":          ("D2/3",    "D2/3\n(FLB)",      "Dopamine"),
    "Dopamine | target-DAT_tracer-fpcit_n-174_dx-hc_pub-dukart2018":            ("DAT",     "DAT\n(FPCIT)",     "Dopamine"),
    "Dopamine | target-FDOPA_tracer-fluorodopa_n-12_dx-hc_pub-garciagomez2018": ("FDOPA",   "FDOPA\n(F18)",     "Dopamine"),
    "GABA | target-GABAa_tracer-flumazenil_n-6_dx-hc_pub-dukart2018":           ("GABAa",   "GABAa\n(flum)",    "GABA"),
    "Glutamate | target-mGluR5_tracer-abp688_n-73_dx-hc_pub-smart2019":         ("mGluR5",  "mGluR5\n(abp)",    "Glutamate"),
    "Glutamate | target-NMDA_tracer-ge179_n-29_dx-hc_pub-galovic2021":          ("NMDA",    "NMDA\n(ge179)",    "Glutamate"),
    "Noradrenaline/Acetylcholine | target-NET_tracer-mrb_n-10_dx-hc_pub-hesse2017":           ("NET",     "NET\n(MRB)",       "NA/ACh"),
    "Noradrenaline/Acetylcholine | target-VAChT_tracer-feobv_n-18_dx-hc_pub-aghourian2017":   ("VAChT",   "VAChT\n(FEOBV)",   "NA/ACh"),
}

SYSTEM_COLORS = {
    "Serotonin":  "#4E9AC7",
    "Dopamine":   "#E07B54",
    "GABA":       "#9B6CB0",
    "Glutamate":  "#5FAD71",
    "NA/ACh":     "#E0A0C0",
}

MAP_ORDER = list(MAP_INFO.keys())

In [ ]:
df = pd.read_csv(DATA_PATH)
df["fz"] = np.arctanh(df["colocalization"].clip(-0.9999, 0.9999))

coloc = pd.read_csv(COLOC_PATH)

def match_full_map(ref_str):
    for full in MAP_ORDER:
        key = full.split("target-")[-1].split("_tracer")[0].lower()
        if key in ref_str.lower():
            return full
    return None

coloc["map_full"] = coloc["reference_map"].apply(match_full_map)
coloc = coloc.dropna(subset=["map_full"])
q_lookup = {(r["map_full"], r["contrast"]): r["q"] for _, r in coloc.iterrows()}

print("Data shape:", df.shape)
print("FZ range:", df["fz"].min().round(3), df["fz"].max().round(3))

sig = [(k, v) for k, v in q_lookup.items() if v < 0.05]
print(f"FDR-significant entries: {len(sig)}")
for k, v in sorted(sig):
    print(f"  {k[0].split('target-')[1].split('_')[0]:10s} | {k[1]:20s} | q={v:.4f}")

In [ ]:
SYSTEM_BANDS = [
    ( 0,  4, "#f0f5fc"),
    ( 5,  8, "#fdf3ee"),
    ( 9,  9, "#f5f0fc"),
    (10, 11, "#eefaf2"),
    (12, 13, "#fceef5"),
]


def sig_star(q):
    if q < 0.001: return "***"
    if q < 0.01:  return "**"
    if q < 0.05:  return "*"
    return ""


def draw_violin_row(ax, df_contrast, contrast, map_order, map_info, system_colors, q_lookup):
    n = len(map_order)
    positions = np.arange(n)
    rng = np.random.default_rng(42)

    for x0, x1, col in SYSTEM_BANDS:
        ax.axvspan(x0 - 0.5, x1 + 0.5, color=col, alpha=1.0, zorder=0, lw=0)

    box_data, colors, sig_flags = [], [], []
    for full_map in map_order:
        short, xlabel, system = map_info[full_map]
        vals = df_contrast.loc[df_contrast["map"] == full_map, "fz"].dropna().values
        box_data.append(vals)
        colors.append(system_colors[system])
        sig_flags.append(q_lookup.get((full_map, contrast), 1.0) < 0.05)

    for pos, is_sig in zip(positions, sig_flags):
        if is_sig:
            ax.axvspan(pos - 0.42, pos + 0.42, color="#FFD700", alpha=0.22, zorder=1, lw=0)

    vp = ax.violinplot(
        box_data,
        positions=positions,
        widths=0.72,
        showmeans=False,
        showmedians=False,
        showextrema=False,
    )
    for body, color in zip(vp["bodies"], colors):
        body.set_facecolor(color)
        body.set_alpha(0.40)
        body.set_edgecolor("none")
        body.set_zorder(2)

    bp = ax.boxplot(
        box_data,
        positions=positions,
        widths=0.12,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color="white", lw=2.5),
        whiskerprops=dict(color="#333333", lw=1.3),
        capprops=dict(color="#333333", lw=1.3),
        boxprops=dict(lw=0),
    )
    for patch in bp["boxes"]:
        patch.set_facecolor("#444444")
        patch.set_alpha(1.0)
        patch.set_zorder(3)

    for pos, full_map, color in zip(positions, map_order, colors):
        vals = df_contrast.loc[df_contrast["map"] == full_map, "fz"].dropna().values
        n_pts = min(200, len(vals))
        idx = rng.choice(len(vals), size=n_pts, replace=False)
        sub = vals[idx]
        jitter = rng.normal(0, 0.09, size=n_pts)
        ax.scatter(pos + jitter, sub, s=4, color=color, alpha=0.25, zorder=4, linewidths=0)

    for i, (pos, full_map) in enumerate(zip(positions, map_order)):
        q = q_lookup.get((full_map, contrast), 1.0)
        star = sig_star(q)
        if star:
            vals = df_contrast.loc[df_contrast["map"] == full_map, "fz"].dropna().values
            cap_top = bp["caps"][2 * i + 1].get_ydata()[0]
            y_star = max(np.percentile(vals, 98), cap_top) + 0.07
            ax.text(pos, y_star, star, ha="center", va="bottom", fontsize=12,
                    fontweight="bold", color="black")

    short_labels = [map_info[m][0] for m in map_order]
    ax.axhline(0, color="#888888", lw=0.8, linestyle="--", alpha=0.6, zorder=1)
    ax.set_xticks(positions)
    ax.set_xticklabels(short_labels, fontsize=8, rotation=0)
    ax.set_xlim(-0.6, n - 0.4)
    ax.set_ylabel("Fisher's z (Spearman ρ)", fontsize=9.5)
    ax.set_title(contrast, fontsize=11, fontweight="bold", loc="left", pad=6)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    ax.grid(axis="y", which="major", alpha=0.2, lw=0.7)

    for b in [5, 9, 10, 12]:
        ax.axvline(b - 0.5, color="#cccccc", lw=0.8, linestyle="-", zorder=2)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True, sharey=False)
fig.suptitle(
    "Single-subject z-score colocalization: Subcortical Volume (Prodromal Subgroups)",
    fontsize=13, fontweight="bold", y=1.01
)

system_spans = [(0, 4.5, "Serotonin"), (5, 8.5, "Dopamine"), (9, 9.5, "GABA"),
                (10, 11.5, "Glutamate"), (12, 13.5, "NA / ACh")]

for i, (contrast, ax) in enumerate(zip(CONTRAST_ORDER, axes)):
    df_c = df[df["contrast"] == contrast]
    draw_violin_row(ax, df_c, contrast, MAP_ORDER, MAP_INFO, SYSTEM_COLORS, q_lookup)
    ax.text(-0.08, 1.02, "abcdefghijklmnopqrstuvwxyz"[i],
            transform=ax.transAxes, fontsize=12,
            fontweight='bold', va='top', clip_on=False)

    if i == 0:
        for x0, x1, label in system_spans:
            ax.annotate(
                "", xy=(x1, 1.08), xytext=(x0, 1.08),
                xycoords=("data", "axes fraction"),
                arrowprops=dict(arrowstyle="-", color="#555555", lw=1.2),
            )
            ax.text(
                (x0 + x1) / 2, 1.10, label,
                transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=9, color="#333333",
                fontweight="semibold",
            )

legend_patches = [
    mpatches.Patch(facecolor=c, alpha=0.75, label=s)
    for s, c in SYSTEM_COLORS.items()
]
legend_patches.append(mpatches.Patch(facecolor="#FFD700", alpha=0.40, label="FDR q < 0.05 (group level)"))
fig.legend(
    handles=legend_patches,
    loc="lower center",
    ncol=6,
    frameon=False,
    fontsize=9,
    bbox_to_anchor=(0.5, -0.03),
)

fig.text(
    0.5, -0.055,
    "* q < 0.05   ** q < 0.01   *** q < 0.001  (FDR-corrected, group-level Hedges' g colocalization)\n"
    "Gold background = FDR-significant map (group level). Violin = full distribution; inner box = IQR + median; points = up to 200 subsampled subjects.\n"
    "y-axis: Fisher's z-transformed per-subject Spearman ρ between individual subcortical volume deviation map and neurotransmitter reference map.",
    ha="center", va="top", fontsize=8, color="#444444", style="italic"
)

plt.tight_layout(h_pad=2.5)
out_path = OUT_DIR / "figure_violin_zscore_volume_subcortical_subgroups.png"
fig.savefig(out_path, dpi=300, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()